In [1]:
import pandas as pd
import numpy as np
import os
from google.cloud import bigquery
from dotenv import load_dotenv
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

# ==========================================
# 1. CONEXIÓN A GOOGLE BIGQUERY
# ==========================================
load_dotenv('../.env')

# IMPORTANTE: Asegúrate de poner el nombre exacto de tu archivo JSON aquí
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../gcp_key.json' 

client = bigquery.Client()

print("Descargando datos para el A/B Test desde BigQuery...")

# Consultamos directamente tu tabla en la nube
query = """
    SELECT campaign_id, impressions, clicks, orders 
    FROM `mktng-performance-dashboard.data.marketing_kpis`
"""
df = client.query(query).to_dataframe()

print("¡Datos listos! Preparando a los competidores...\n")

# ==========================================
# 2. PREPARACIÓN DE LOS DATOS (El código original se mantiene)
# ==========================================
# Agrupar los datos por campaña para tener los totales
df_grouped = df.groupby('campaign_id')[['impressions', 'clicks', 'orders']].sum().reset_index()

# Ordenar por impresiones para elegir las dos campañas más grandes
df_grouped = df_grouped.sort_values(by='impressions', ascending=False).head(2)

# Mostrar la tabla en pantalla para verificar los datos seleccionados
display(df_grouped)

# Extraer los datos de las dos campañas (Campaña A y Campaña B)
camp_a_id = df_grouped.iloc[0]['campaign_id']
camp_b_id = df_grouped.iloc[1]['campaign_id']

impresiones = [df_grouped.iloc[0]['impressions'], df_grouped.iloc[1]['impressions']]
clics = [df_grouped.iloc[0]['clicks'], df_grouped.iloc[1]['clicks']]
ordenes = [df_grouped.iloc[0]['orders'], df_grouped.iloc[1]['orders']]

print(f"--- ENFRENTAMIENTO: Campaña {camp_a_id} vs Campaña {camp_b_id} ---\n")

Descargando datos para el A/B Test desde BigQuery...
¡Datos listos! Preparando a los competidores...



,campaign_id,impressions,clicks,orders
1,39889,1068337427,420003,1566
7,983498,137806768,509992,313


--- ENFRENTAMIENTO: Campaña 39889 vs Campaña 983498 ---

CTR Campaña A: 0.04% | CTR Campaña B: 0.37%
P-Valor del Z-Test: 0.0000
✅ Resultado: Con un 95% de confianza, la Campaña B es superior en generar Clics.



In [ ]:
# ==========================================
# PRUEBA 1: Z-Test para el CTR (Click-Through Rate)
# ==========================================
stat_z, p_valor_z = proportions_ztest(count=clics, nobs=impresiones)

ctr_a = (clics[0] / impresiones[0]) * 100
ctr_b = (clics[1] / impresiones[1]) * 100

print(f"CTR Campaña A: {ctr_a:.2f}% | CTR Campaña B: {ctr_b:.2f}%")
print(f"P-Valor del Z-Test: {p_valor_z:.4f}")

if p_valor_z < 0.05:
    ganador = "Campaña A" if ctr_a > ctr_b else "Campaña B"
    print(f"✅ Resultado: Con un 95% de confianza, la {ganador} es superior en generar Clics.\n")
else:
    print("⚖️ Resultado: No hay diferencia estadística significativa en el CTR. Fue casualidad.\n")

In [ ]:
# ==========================================
# PRUEBA 2: Chi-Cuadrado para Tasa de Conversión (CR)
# ==========================================
no_ordenes_a = clics[0] - ordenes[0]
no_ordenes_b = clics[1] - ordenes[1]

tabla_contingencia = np.array([
    [ordenes[0], no_ordenes_a],
    [ordenes[1], no_ordenes_b]
])

chi2, p_valor_chi2, dof, expected = chi2_contingency(tabla_contingencia)

cr_a = (ordenes[0] / clics[0]) * 100
cr_b = (ordenes[1] / clics[1]) * 100

print(f"Conversion Rate Campaña A: {cr_a:.2f}% | Conversion Rate Campaña B: {cr_b:.2f}%")
print(f"P-Valor del Chi-Cuadrado: {p_valor_chi2:.4f}")

if p_valor_chi2 < 0.05:
    ganador_cr = "Campaña A" if cr_a > cr_b else "Campaña B"
    print(f"✅ Resultado: Con un 95% de confianza, la {ganador_cr} es superior en generar Ventas.")
else:
    print("⚖️ Resultado: No hay diferencia estadística significativa en la Conversión.")